# AI알고리즘 2주차, 데이터 과학의 이해와 분석
전인성, 광주교육대학교 컴퓨터교육과

교재 2027 시나공 AI 능력시험 AICE ASSOCIATE 1~6장과 연결되는 수업용 실습입니다. 예시 코드와 항공권 12건은 이 수업을 위해 새로 작성한 가상 자료입니다. 실제 항공사나 요금을 나타내지 않습니다.

코랩에서 Drive에 사본 저장 후 위에서 아래로 실행하세요. 외부 CSV 업로드는 필요하지 않습니다. 런타임 초기화 후 준비 셀부터 다시 실행합니다.

## 시작 질문
비행시간이 길면 가격도 높을까요? 예상과 이유를 먼저 기록하세요.

In [ ]:
prediction = ''  # 나의 예상과 이유

## 1장, 작업 환경
Shift + Enter로 실행합니다. 라이브러리와 버전을 확인하세요.

In [ ]:
import sys, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
print('Python', sys.version.split()[0], 'pandas', pd.__version__)

## 2장, 데이터 획득
NumPy는 원소별 연산을 지원합니다. 가격은 천 원, 비행시간은 시간, 남은 일수는 일입니다. id는 식별자입니다.

In [ ]:
prices = np.array([60, 80, 100])
print(prices * 0.9)
print(pd.Series(prices, name='price'))

In [ ]:
csv_text = 'id,airline,seat,duration,days_left,price\n101,가람,일반,1,21,60\n102,누리,일반,1.5,14,80\n103,가람,일반,2,7,100\n104,누리,일반,2.5,3,120\n105,가람,일반,3,1,140\n106,누리,일반,,10,90\n107,가람,우등,1,21,180\n108,누리,우등,1.5,14,200\n109,가람,우등,2,7,220\n110,누리,우등,2.5,3,240\n111,가람,우등,3,1,260\n112,누리,우등,4,2,900\n'
df = pd.read_csv(io.StringIO(csv_text))
original = df.copy()
display(df)
df.to_csv('week2_flights.csv', index=False, encoding='utf-8-sig')
reloaded = pd.read_csv('week2_flights.csv')
assert reloaded.shape == (12, 6)

## 3장, 구조와 요약
shape와 info를 비교해 결측값을 찾아보세요. head의 일부 행만으로 전체를 일반화하지 않습니다.

In [ ]:
print(df.shape)
print(df.columns.tolist())
df.info()
display(df.head(3), df.tail(3))
display(df[['duration','days_left','price']].describe())
display(df['airline'].value_counts())

## 4장, 선택과 변경
loc는 레이블, iloc는 위치로 선택합니다. 두 결과가 같은지 확인하세요.

In [ ]:
labeled = df.set_index('id')
a = labeled.loc[101:103, ['airline','price']]
b = labeled.iloc[0:3, [0,4]]
assert a.equals(b)
display(a)

In [ ]:
max_price = 200  # 값을 바꾸어 비교
selected = df[(df['seat']=='일반') & (df['price']<=max_price)]
display(selected)
print('건수', len(selected))
work = df.copy()
work['discount_price'] = work['price'] * 0.9
work = work.rename(columns={'seat':'seat_type'})
work = work.drop(columns=['discount_price'])
display(work.sort_values('price', ascending=False))

### 그룹화, 피벗과 다중 인덱스
평균과 개수를 함께 봅니다. 중복 조합은 pivot_table로 집계하세요.

In [ ]:
display(df.groupby('airline')['price'].agg(['mean','median','count']))
wide = df.pivot_table(index='airline', columns='seat', values='price', aggfunc='mean')
summary = df.groupby(['airline','seat'])['price'].mean()
display(wide, summary, summary.unstack('seat'), wide.stack())

### 표 연결
concat은 이어 붙이고 merge는 공통 키를 사용합니다. validate로 오른쪽 키가 유일한지 확인합니다.

In [ ]:
combined = pd.concat([df.iloc[:6],df.iloc[6:]], ignore_index=True)
assert len(combined)==12
airlines = pd.DataFrame({'airline':['가람','누리'],'category':['A','B']})
display(df.merge(airlines, on='airline', how='left', validate='many_to_one'))
display(df.set_index('airline').join(airlines.set_index('airline'), how='left').head())

In [ ]:
tickets = pd.DataFrame({'id':[1,2], 'airline':['가람','누리']})
lookup = pd.DataFrame({'airline':['가람','다온'], 'category':['A','B']})
for how in ['inner','left','right','outer']:
    print(how)
    display(tickets.merge(lookup, on='airline', how=how))

## 5장, 데이터 이해
900천 원 포함 여부에 따라 평균과 중앙값을 비교하세요. 이것만으로 삭제가 옳다고 판단하지 않습니다.

In [ ]:
for include in [True,False]:
    sample = df if include else df[df['price']!=900]
    print('900 포함',include,'건수',len(sample),'평균',sample['price'].mean(),'중앙값',sample['price'].median())
display(pd.crosstab(df['airline'],df['seat']))

그래프는 글꼴과 무관하게 읽도록 영문 범례를 씁니다. Economy는 일반, Premium은 우등입니다.

In [ ]:
plot_df = df.assign(seat_en=df['seat'].map({'일반':'Economy','우등':'Premium'}), airline_en=df['airline'].map({'가람':'Garam','누리':'Nuri'}))
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist(df['price'], bins=6, range=(0,900), edgecolor='white')
axes[0].set(xlabel='Price (thousand KRW)',ylabel='Count')
sns.scatterplot(data=plot_df,x='duration',y='price',hue='seat_en',ax=axes[1])
axes[1].set(xlabel='Duration (hours)',ylabel='Price (thousand KRW)')
plt.tight_layout();plt.show()

In [ ]:
display(df[['duration','days_left','price']].corr())
for seat,g in df.groupby('seat'):
    print(seat,'유효 쌍',len(g[['duration','price']].dropna()),'상관',g['duration'].corr(g['price']))
sns.heatmap(df[['duration','days_left','price']].corr(),annot=True,vmin=-1,vmax=1,cmap='coolwarm')
plt.show()

### 시각화 확장
범주별 건수, 분포, 회귀선과 두 변수의 분포를 비교합니다. 회귀선은 인과관계의 증거가 아닙니다.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
sns.countplot(data=plot_df,x='airline_en',hue='seat_en',ax=axes[0])
sns.boxplot(data=plot_df,x='seat_en',y='price',ax=axes[1])
plt.tight_layout();plt.show()
sns.lmplot(data=plot_df.dropna(subset=['duration']),x='duration',y='price',hue='seat_en',ci=None)
plt.show()
sns.jointplot(data=plot_df.dropna(subset=['duration']),x='duration',y='price')
plt.show()

## 6장, 전처리
매번 원본을 복사해 결측 처리 방법을 비교합니다. 전체 표를 사용하는 수업용 탐색이며 성능 평가용 전처리는 아래 확장 예를 따릅니다.

In [ ]:
display(df.isna().sum())
for method in ['drop','mean','median']:
    clean=original.copy()
    if method=='drop':
        clean=clean.dropna(subset=['duration'])
    else:
        value=getattr(clean['duration'],method)()
        clean['duration']=clean['duration'].fillna(value)
    print(method,'행 수',len(clean),'결측',clean['duration'].isna().sum(),'평균',clean['duration'].mean())

### 이상치 점검
IQR은 점검 기준입니다. 삭제나 대체는 자료 생성 배경과 분석 목적을 확인한 뒤 결정합니다.

In [ ]:
q1,q3=df['price'].quantile([.25,.75])
iqr=q3-q1
low,high=q1-1.5*iqr,q3+1.5*iqr
print('Q1,Q3,IQR,하한,상한',q1,q3,iqr,low,high)
display(df[(df['price']<low)|(df['price']>high)])
display(df.assign(price_clipped=df['price'].clip(low,high))[['id','price','price_clipped']])

### 구간화와 인코딩
2시간은 첫 구간에 포함됩니다. 숫자 범주 코드의 간격에 의미를 부여하지 않습니다.

In [ ]:
display(pd.cut(df['duration'], bins=[0,2,4],labels=['2시간 이하','2시간 초과'],include_lowest=True))
display(pd.qcut(df['price'],q=4,labels=['Q1','Q2','Q3','Q4']))
codes,categories=pd.factorize(df['airline'])
print(categories.tolist(),codes)
display(pd.get_dummies(df[['airline','seat']],dtype=int))

### 스케일링
표준화는 정규분포를 보장하지 않습니다. StandardScaler는 ddof=0을 사용합니다.

In [ ]:
x=pd.Series([10,20,30,40,50],dtype=float)
minmax=(x-x.min())/(x.max()-x.min())
standard=(x-x.mean())/x.std(ddof=0)
display(pd.DataFrame({'original':x,'minmax':minmax,'standard':standard}))
assert np.isclose(standard.mean(),0)
assert np.isclose(standard.std(ddof=0),1)

### 변수 생성과 선택
가격 예측의 입력에서 가격 자체와 식별자를 제외합니다. RFE는 중요도가 낮은 변수를 반복 제거하고 RFECV는 교차 검증을 더합니다. 일변량 선택은 각 변수와 목표값의 관계를 평가합니다. 실행은 모델링 학습과 연결합니다.

In [ ]:
features=df.copy()
features['is_soon']=(features['days_left']<=3).astype(int)
X=features[['duration','days_left','airline','seat']]
y=features['price']
display(features[['id','days_left','is_soon']])

### 확장, 학습 자료에서만 전처리 기준 구하기
누수 없는 순서의 예입니다. 12건으로 모델 성능을 평가하지 않습니다. 학습 자료에서 fit하고 평가 자료에는 transform만 적용합니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42)
num=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
preprocess=ColumnTransformer([('num',num,['duration','days_left']),('cat',OneHotEncoder(handle_unknown='ignore'),['airline','seat'])])
train_ready=preprocess.fit_transform(X_train)
test_ready=preprocess.transform(X_test)
print('학습 행렬',train_ready.shape,'평가 행렬',test_ready.shape)
assert train_ready.shape[1]==test_ready.shape[1]

## 통합 실습과 성찰
좌석별 가격 비교와 900천 원 관측값의 영향을 설명하세요. 결측 처리 방법을 선택하고 그 근거, 처음 예상에서 바뀐 점, 한계와 추가 질문을 적으세요. 결과 파일은 학교 이러닝에 직접 제출합니다.

In [ ]:
finding='' # 자료에서 확인한 결과
reason='' # 전처리 선택과 근거
limitation='' # 한계와 추가 질문
report=f'AI알고리즘 2주차 분석 기록\n\n예상\n{prediction}\n\n결과\n{finding}\n\n처리 이유\n{reason}\n\n한계\n{limitation}\n'
with open('week2_reflection.txt','w',encoding='utf-8-sig') as f:
    f.write(report)
print(report)

왼쪽 파일 목록에서 week2_reflection.txt를 다운로드하세요. 코드로 내려받으려면 아래 두 줄의 주석을 해제합니다.

In [ ]:
# from google.colab import files
# files.download('week2_reflection.txt')

## 참고
KT 서길원 외(2026), 2027 시나공 AI 능력시험 AICE ASSOCIATE, 길벗, 1~6장(18~163쪽).

[pandas 선택](https://pandas.pydata.org/docs/user_guide/indexing.html), [병합](https://pandas.pydata.org/docs/user_guide/merging.html), [scikit-learn 데이터 누수](https://scikit-learn.org/stable/common_pitfalls.html), [Colab FAQ](https://research.google.com/colaboratory/faq.html).